# Abstract Factory Design Pattern 

explained using your specific Logistics (Vehicle + Navigation) example.

#### The Concept

The Abstract Factory is a "Factory of Factories". It allows you to create **families of related objects** without specifying their concrete classes. 

**The Rule**: `A Truck` must always go with `GPS`. A `Ship` must always go with `Sonar`. You cannot mix them (e.g., A Ship with a GPS might get lost at sea). The Abstract Factory ensures these families stay together.

## The Classic OOP Way (Java-Style)

In the strict OOP approach, we use **Abstract Base Classes (Interfaces) for everything**: the products (`Vehicle`, `Navigation`) and the factory (`TransportFactory`). This ensures compile-time safety (in statically typed languages) and strict structure.

#### THE ABSTRACT PRODUCTS (Interfaces)

These interfaces define the "Contract". Any new vehicle or navigation system </br>
added later must adhere to these rules. The client code only talks to these. </br> 

In [4]:
from abc import ABC, abstractmethod

class Vehicle(ABC):
    """
    Abstract Product A: Defines the interface for all vehicle types.
    """
    @abstractmethod
    def move_cargo(self) -> str:
        pass

class Navigation(ABC):
    """
    Abstract Product B: Defines the interface for all navigation controls.
    """
    @abstractmethod
    def calculate_route(self) -> str:
        pass

#### THE CONCRETE PRODUCT FAMILIES

These are the actual implementations. Note that we have two distinct families:
- The Road Family (Truck + GPS)
- The Sea Family (Ship + Sonar)

In [5]:
# --- Family 1: Road Implementation ---
class Truck(Vehicle):
    def move_cargo(self) -> str:
        return "Truck: Driving on highway with 18 wheels."

class GPS(Navigation):
    def calculate_route(self) -> str:
        return "GPS: Calculating overload road path via satellites."

# --- Family 2: Sea Implementation ---
class Ship(Vehicle):
    def move_cargo(self) -> str:
        return "Ship: Sailing across the Atlantic ocean."

class Sonar(Navigation):
    def calculate_route(self) -> str:
        return "Sonar: Checking water depth and coral reefs."

#### THE ABSTRACT FACTORY (The Blueprint)

In [6]:
class TransportFactory(ABC):
    """
    The Abstract Factory Interface.
    
    This is the core of the pattern. It declares a set of methods that return
    different abstract products. 
    
    Crucial Point: It forces any concrete factory to create *both* a Vehicle 
    and a Navigation system, ensuring they always come in pairs.
    """
    @abstractmethod
    def create_vehicle(self) -> Vehicle:
        pass

    @abstractmethod
    def create_navigation(self) -> Navigation:
        pass

#### THE CONCRETE FACTORIES (The Manufacturers)

These classes implement the creation logic. Each factory is responsible for
creating one specific variant of the product family.

In [7]:
class RoadFactory(TransportFactory):
    """
    Concrete Factory 1: Produces only Road-compatible objects.
    Guarantees that if you get a Vehicle from here, it's a Truck.
    """
    def create_vehicle(self) -> Vehicle:
        return Truck()

    def create_navigation(self) -> Navigation:
        return GPS()

class SeaFactory(TransportFactory):
    """
    Concrete Factory 2: Produces only Sea-compatible objects.
    Guarantees that if you get a Vehicle from here, it's a Ship.
    """
    def create_vehicle(self) -> Vehicle:
        return Ship()

    def create_navigation(self) -> Navigation:
        return Sonar()

#### CLIENT CODE (The Application)

In [8]:
class LogisticsApplication:
    """
    The Client. 
    
    NOTICE: This class knows NOTHING about 'Truck', 'Ship', 'RoadFactory', etc.
    It only knows about the abstract types 'TransportFactory', 'Vehicle', and 
    'Navigation'.
    
    This is 'Dependency Injection' - we inject the factory we want to use.
    """
    def __init__(self, factory: TransportFactory):
        self.factory = factory

    def start_mission(self) -> None:
        # 1. Create the objects using the injected factory
        vehicle: Vehicle = self.factory.create_vehicle()
        nav: Navigation = self.factory.create_navigation()

        # 2. Use the objects (The logic remains the same regardless of factory type)
        print(f"Vehicle Status: {vehicle.move_cargo()}")
        print(f"Nav Status:     {nav.calculate_route()}")
        print("-" * 50)

#### EXECUTION

In [9]:
def main():
    # Imagine this setting is read from a 'config.json' file or environment variable
    # Options: "road" or "sea"
    current_os_config = "sea" 

    print(f"--- Booting Logistics App (Mode: {current_os_config.upper()}) ---\n")

    # The Factory Selection Logic
    # This is usually the ONLY place in your code that knows about concrete classes.
    selected_factory: TransportFactory

    match current_os_config:
        case "road":
            selected_factory = RoadFactory()
        case "sea":
            selected_factory = SeaFactory()
        case _:
            raise ValueError("Unknown configuration type")

    # Pass the factory to the application.
    # The app works automatically with whatever factory we gave it.
    app = LogisticsApplication(selected_factory)
    app.start_mission()

if __name__ == "__main__":
    main()

--- Booting Logistics App (Mode: SEA) ---

Vehicle Status: Ship: Sailing across the Atlantic ocean.
Nav Status:     Sonar: Checking water depth and coral reefs.
--------------------------------------------------


## The Pythonic Way

In Python, classes are objects. We don't need to create `RoadFactory` and `SeaFactory` classes just to return other classes. We can simply:
- Group the classes (families) into **Tuples** or a **Dictionary**.
- Pass the class references directly to the application.
- The Application acts as the factory, instantiating what it was given.

This removes the boilerplate of the `TransportFactory`, `RoadFactory`, and `SeaFactory` classes entirely while keeping the safety of the pattern.

#### THE PRODUCTS (Classes)

In [10]:
class Truck:
    def move(self): return "🚚 Truck: Driving on highway"

class GPS:
    def route(self): return "🛰️ GPS: Calculating road path"

class Ship:
    def move(self): return "🚢 Ship: Sailing across ocean"

class Sonar:
    def route(self): return "📡 Sonar: Checking depth"

#### THE PYTHONIC "FACTORIES" (Just Configuration)

We define families using a dictionary mapping strings to Tuples of Classes. This replaces the entire hierarchy of Factory classes.

In [11]:
FACTORIES = {
    "road": (Truck, GPS),
    "sea":  (Ship, Sonar)
}

#### THE CLIENT

In [13]:
from typing import Type

class LogisticsApp:
    def __init__(self, vehicle_cls: Type, nav_cls: Type):
        """
        Expects class references (e.g., Truck), not instances.
        """
        self.vehicle = vehicle_cls()
        self.navigation = nav_cls()

    def run(self):
        print(self.navigation.route())
        print(self.vehicle.move())

#### MAIN

In [14]:
def main():
    mode = "sea" # Imagine this comes from a config file

    # 1. Retrieve the family of classes
    if mode not in FACTORIES:
        raise ValueError("Unknown mode")
    
    # Unpack the tuple (VehicleClass, NavigationClass)
    v_cls, n_cls = FACTORIES[mode]

    print(f"--- Starting {mode.upper()} Logistics ---")
    
    # 2. Inject them into the app
    app = LogisticsApp(v_cls, n_cls)
    app.run()

if __name__ == "__main__":
    main()

--- Starting SEA Logistics ---
📡 Sonar: Checking depth
🚢 Ship: Sailing across ocean


#### Key Differences

| Feature               | Classic OOP                                                      | Pythonic                                                     |
|-----------------------|------------------------------------------------------------------|---------------------------------------------------------------|
| **Factory Structure** | Explicit `RoadFactory` class implementing `TransportFactory`.   | A dictionary or tuple (e.g., `(Truck, GPS)`).                  |
| **Instantiation**     | Done inside factory class methods.                               | Done by the client (or helper) using the class reference `v_cls()`. |
| **Boilerplate**       | High — extra classes (`TransportFactory`, `RoadFactory`, `SeaFactory`) just for grouping. | Low — grouping handled via simple data structures.            |


#### When to use which?

- **Java Way**: Use this if your products need complex initialization logic (e.g., `Truck` needs `engine_type` and `GPS` needs api_key which are stored inside the `RoadFactory`).
- **Pythonic Way**: Use this when the products are simple to instantiate or when you want a configuration-driven approach (mapping strings to class families).

# More Pythonic Way

Yes. The previous examples followed the `"Classic" (Gang of Four)` implementation. While correct, it is often criticized in Python for being too verbose and "Java-like" (too many classes, heavy inheritance).

The Pythonic way leverages the fact that Classes and Functions are first-class citizens. You don't need a Factory Class to create objects; you can pass the Class types themselves or use simple functions.

Here is the Modern Pythonic Approach using Protocols (structural typing) and Functional Factories.

The Pythonic Changes
- `Protocol` instead of `ABC`: We don't force classes to inherit from specific parents. As long as they have the right methods (Duck Typing), they work.
- `dict` instead of `Factory Classes`: We map the logic in a simple dictionary.
- Classes as Data: We pass the class Truck directly, rather than making a method `create_truck()`.


#### Why `frozen=True`?
In the `@dataclass(frozen=True)` decorator, the frozen parameter makes the object immutable (unchangeable) after it is created.

#### We use it here for three key reasons:
- Safety (Configuration Integrity): A Factory definition is like a rulebook. `"Road"` implies Truck and GPS. You don't want a buggy part of your code to accidentally overwrite `family.vehicle_class = Ship` later in the execution. If you try to change a frozen object, Python raises a `FrozenInstanceError`.
- Intent: It signals to other developers: "This object is just a container for constant data. Read it, but don't touch it."
- Hashability: Because it is immutable, a frozen dataclass can be used as a key in a dictionary or added to a set (though in this specific example, we use it as a value).

### PROTOCOLS (The "Duck Typing" Interface)

We use Protocols instead of Abstract Base Classes. </br>
This means classes don't need to inherit from anything; </br>
they just need to have the matching methods.</br>

In [11]:
from typing import Protocol, runtime_checkable

@runtime_checkable
class Vehicle(Protocol):
    def move_cargo(self) -> str: ...

@runtime_checkable
class Navigation(Protocol):
    def calculate_route(self) -> str: ...


### CONCRETE CLASSES (The Implementation)

These are standalone classes. They are NOT instantiated here.

In [12]:
class Truck:
    def move_cargo(self) -> str: 
        return "Truck driving on highway"

class GPS:
    def calculate_route(self) -> str: 
        return "GPS calculating path via satellite"

class Ship:
    def move_cargo(self) -> str: 
        return "Ship sailing on waves"

class Sonar:
    def calculate_route(self) -> str: 
        return "Sonar checking water depth"

### THE PYTHONIC FACTORY (Registry)

In [13]:
@dataclass(frozen=True)
class TransportFamily:
    """
    A strictly typed container that holds the class references.
    frozen=True prevents accidental changes to the factory rules.
    """
    vehicle_class: Type[Vehicle]  # Stores the Class itself, not an instance
    nav_class: Type[Navigation]

# The Registry: Maps a string key to a family of classes.
# This replaces the need for "RoadFactory" and "SeaFactory" classes.
FACTORIES = {
    "road": TransportFamily(vehicle_class=Truck, nav_class=GPS),
    "sea":  TransportFamily(vehicle_class=Ship,  nav_class=Sonar),
}

def get_factory(mode: str) -> TransportFamily:
    """Retrieves the correct family based on configuration string."""
    try:
        return FACTORIES[mode]
    except KeyError:
        raise ValueError(f"Unknown mode: {mode}")

### CLIENT CODE

In [14]:
from typing import Protocol, Type, runtime_checkable
from dataclasses import dataclass


def run_mission(family: TransportFamily):
    """
    The client code receives the factory family.
    It performs the instantiation (adding parentheses) here.
    """
    print(f"--- Initializing Mission ---")
    
    # INSTANTIATION HAPPENS HERE
    vehicle_instance = family.vehicle_class() 
    nav_instance = family.nav_class()

    # USAGE
    print(f"Vehicle: {vehicle_instance.move_cargo()}")
    print(f"Nav:     {nav_instance.calculate_route()}")
    print("-" * 30)

def main():
    # Scenario 1: Road
    print("User configures: ROAD")
    road_tools = get_factory("road")
    run_mission(road_tools)

    # Scenario 2: Sea
    print("User configures: SEA")
    sea_tools = get_factory("sea")
    run_mission(sea_tools)

if __name__ == "__main__":
    main()

User configures: ROAD
--- Initializing Mission ---
Vehicle: Truck driving on highway
Nav:     GPS calculating path via satellite
------------------------------
User configures: SEA
--- Initializing Mission ---
Vehicle: Ship sailing on waves
Nav:     Sonar checking water depth
------------------------------


# Abstract Factory Design Pattern 

explained using a complex, real-world scenario: **A Multi-Cloud Infrastructure Provisioning Tool** (like Terraform or Pulumi).

#### The Scenario: Cross-Cloud Deployment
You are building a tool that allows users to deploy their application stack to either **AWS** or **Google Cloud (GCP)**. A "Stack" always consists of two things that must work together:

- **Compute Instance**: (AWS uses **EC2**, GCP uses **Compute Engine**).
- **Object Storage**: (AWS uses **S3**, GCP uses **Cloud Storage**).

**The Constraint**: You cannot mix them. You can't create an AWS EC2 instance and try to attach a Google Cloud Storage bucket to it using internal private networking. They must be created as a matching family.

## The Classic OOP Way (Java-Style)

We define interfaces for the products (`ICompute`, `IStorage`) and the factory (`ICloudFactory`). We create specific factory classes (`AWSFactory`, `GCPFactory`) that hardcode the relationship between the products.

#### ABSTRACT PRODUCTS (Interfaces)

In [15]:
from abc import ABC, abstractmethod

class ICompute(ABC):
    @abstractmethod
    def start(self): pass

class IStorage(ABC):
    @abstractmethod
    def create_bucket(self, name: str): pass

#### CONCRETE PRODUCTS (AWS Family)

In [16]:
class EC2Instance(ICompute):
    def start(self):
        print("☁️ AWS: Starting EC2 t2.micro instance...")

class S3Storage(IStorage):
    def create_bucket(self, name: str):
        print(f"📦 AWS: Creating S3 Bucket '{name}'")

#### CONCRETE PRODUCTS (GCP Family)

In [17]:
class GCEInstance(ICompute):
    def start(self):
        print("☁️ GCP: Starting Compute Engine f1-micro...")

class GCSStorage(IStorage):
    def create_bucket(self, name: str):
        print(f"📦 GCP: Creating GCS Bucket '{name}'")

#### ABSTRACT FACTORY

In [18]:
from abc import ABC, abstractmethod

class ICloudFactory(ABC):
    @abstractmethod
    def create_compute(self) -> ICompute: pass

    @abstractmethod
    def create_storage(self) -> IStorage: pass

#### CONCRETE FACTORIES

In [19]:
class AWSFactory(ICloudFactory):
    def create_compute(self) -> ICompute:
        return EC2Instance()

    def create_storage(self) -> IStorage:
        return S3Storage()

class GCPFactory(ICloudFactory):
    def create_compute(self) -> ICompute:
        return GCEInstance()

    def create_storage(self) -> IStorage:
        return GCSStorage()

#### CLIENT CODE (Deployment Manager)

In [20]:
class DeploymentManager:
    def __init__(self, factory: ICloudFactory):
        # The manager doesn't know if it's AWS or GCP.
        # It just knows it has a valid factory.
        self.compute = factory.create_compute()
        self.storage = factory.create_storage()

    def deploy_stack(self, app_name: str):
        print(f"--- Deploying '{app_name}' ---")
        self.storage.create_bucket(f"{app_name}-logs")
        self.compute.start()
        print("✅ Stack deployed successfully.\n")

#### MAIN

def main():
    # User selects AWS
    aws_factory = AWSFactory()
    app1 = DeploymentManager(aws_factory)
    app1.deploy_stack("my-shop-app")

    # User selects GCP
    gcp_factory = GCPFactory()
    app2 = DeploymentManager(gcp_factory)
    app2.deploy_stack("my-data-pipeline")

if __name__ == "__main__":
    main()

## The Pythonic Way (Data-Driven Configuration)

In the OOP version, adding Azure would require creating 3 new classes (`AzureCompute`, `AzureStorage`, `AzureFactory`). In Python, we can simplify this by treating classes as data. We can use a Configuration Dictionary to map the provider string (e.g., `"aws"`) directly to the tuple of classes required.

This removes the need for the `Factory` classes entirely. The "Abstract Factory" becomes just a dictionary lookup.

#### THE PRODUCTS (Classes)

In [22]:
from typing import Type, Protocol

# We use Protocols (Python's Duck Typing Interface) for type hinting
class ComputeProto(Protocol):
    def start(self): ...

class StorageProto(Protocol):
    def create_bucket(self, name: str): ...

# --- AWS ---
class EC2:
    def start(self): print("☁️ AWS: EC2 Instance started.")
class S3:
    def create_bucket(self, name): print(f"📦 AWS: S3 Bucket '{name}' created.")

# --- GCP ---
class GCE:
    def start(self): print("☁️ GCP: Compute Engine started.")
class GCS:
    def create_bucket(self, name): print(f"📦 GCP: Cloud Storage '{name}' created.")

# --- Azure (Easy to add!) ---
class AzureVM:
    def start(self): print("☁️ Azure: VM started.")
class AzureBlob:
    def create_bucket(self, name): print(f"📦 Azure: Blob Container '{name}' created.")

#### THE PYTHONIC ABSTRACT FACTORY (Registry)

In [23]:
# Instead of Factory Classes, we map the Provider ID to a Tuple of Classes.
# Structure: "provider": (ComputeClass, StorageClass)
CLOUD_PROVIDERS = {
    "aws":   (EC2, S3),
    "gcp":   (GCE, GCS),
    "azure": (AzureVM, AzureBlob)
}

#### CLIENT CODE

In [24]:
class CloudDeployer:
    def __init__(self, provider_name: str):
        # 1. Factory Logic: Lookup the family
        if provider_name not in CLOUD_PROVIDERS:
            raise ValueError(f"Unknown provider: {provider_name}")
        
        # 2. Unpack the classes (Not instances yet)
        compute_cls, storage_cls = CLOUD_PROVIDERS[provider_name]
        
        # 3. Instantiate
        self.compute = compute_cls()
        self.storage = storage_cls()

    def deploy(self, project_name: str):
        print(f"--- Deploying to {self.compute.__class__.__name__[:3]} ---")
        self.storage.create_bucket(f"{project_name}-assets")
        self.compute.start()

#### MAIN

In [25]:
def main():
    # Configuration usually comes from env variables or yaml
    user_choice = "azure" 

    deployer = CloudDeployer(user_choice)
    deployer.deploy("enterprise-portal")

    # Switching is trivial
    deployer_aws = CloudDeployer("aws")
    deployer_aws.deploy("startup-mvp")

if __name__ == "__main__":
    main()

--- Deploying to Azu ---
📦 Azure: Blob Container 'enterprise-portal-assets' created.
☁️ Azure: VM started.
--- Deploying to EC2 ---
📦 AWS: S3 Bucket 'startup-mvp-assets' created.
☁️ AWS: EC2 Instance started.


#### Key Differences

| Feature          | Classic OOP                                                                 | Pythonic                                                                 |
|------------------|------------------------------------------------------------------------------|---------------------------------------------------------------------------|
| **Enforcement**  | Strong — interfaces (`ICompute`) and factory classes prevent mixing families (e.g., EC2 with GCS). | Flexible — the `CLOUD_PROVIDERS` dictionary keeps related families together. |
| **Boilerplate**  | High — requires `AWSFactory`, `GCPFactory`, `AzureFactory` classes just to return new objects. | Low — no factory classes; only a mapping in a dictionary.                 |
| **Extensibility**| Adding Azure requires multiple new classes and client import updates.       | Adding Azure requires product classes plus a single line in `CLOUD_PROVIDERS`. |


#### When to use which?

- **Java Way**: Use this if the creation logic is asymmetric. For example, if creating an `AWS` `EC2` requires an API Key, but creating a `Google` `GCE` requires a JSON Key File path. The `AWSFactory` class gives you a place to store that specific configuration state.
- **Pythonic Way**: Use this for **Symmetric** creation (all classes take similar init arguments) or when you want a plugin-style architecture where providers are just entries in a configuration table.